In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import json
import os
import random
import re
from collections import Counter
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/nlpcc/data")
OUT_DIR = Path("/content/drive/MyDrive/nlpcc/output/track2")
OUT_DIR.mkdir(parents = True, exist_ok=True)

TRAIN_DIR = DATA_DIR / "train.jsonl"
DEV_DIR = DATA_DIR / "dev.jsonl"

print("Train file dir: ", TRAIN_DIR)
print("Dev file dir: ", DEV_DIR)
print("track2 output dir: ", OUT_DIR)

Train file dir:  /content/drive/MyDrive/nlpcc/data/train.jsonl
Dev file dir:  /content/drive/MyDrive/nlpcc/data/dev.jsonl
track2 output dir:  /content/drive/MyDrive/nlpcc/output/track2


In [ ]:
# Cell 2: 官方 19 类 Schwartz 基本人类价值观定义

VALUE_DEFINITIONS = {
    "Self-direction–thought": "Freedom to cultivate one's own ideas and abilities.",
    "Self-direction–action": "Freedom to determine one's own actions.",
    "Stimulation": "Excitement, novelty, and change.",
    "Hedonism": "Pleasure and sensuous gratification.",
    "Achievement": "Success according to social standards.",
    "Power–dominance": "Power through exercising control over people.",
    "Power–resources": "Power through control of material and social resources.",
    "Face": "Security and power through maintaining one's public image and avoiding humiliation.",
    "Security–personal": "Safety in one's immediate environment.",
    "Security–societal": "Safety and stability in the wider society.",
    "Tradition": "Maintaining and preserving cultural, family, or religious traditions.",
    "Conformity–rules": "Compliance with rules, laws, and formal obligations.",
    "Conformity–interpersonal": "Avoidance of upsetting or harming other people.",
    "Humility": "Recognizing one's insignificance in the larger scheme of things.",
    "Benevolence–dependability": "Being a reliable and trustworthy member of the ingroup.",
    "Benevolence–caring": "Devotion to the welfare of ingroup members.",
    "Universalism–concern": "Commitment to equality, justice, and protection for all people.",
    "Universalism–nature": "Preservation of the natural environment.",
    "Universalism–tolerance": "Acceptance and understanding of those who are different from oneself.",
}

LABELS = list(VALUE_DEFINITIONS.keys())
len(LABELS), LABELS[:5]

(19,
 ['Self-direction–thought',
  'Self-direction–action',
  'Stimulation',
  'Hedonism',
  'Achievement'])

In [ ]:
# Cell 3: JSONL 读取与写入

def read_jsonl(path):
  rows = []
  with open(path, "r", encoding = "utf-8") as f:
    for line in f:
      line = line.strip()
      if line:
        rows.append(json.loads(line))
  return rows
def write_jsonl(path, rows):
  path = Path(path)
  path.parent.mkdir(parents=True, exist_ok = True)
  with open(path, "w", encoding = "utf-8") as f:
    for row in rows:
      f.write(json.dumps(row, ensure_ascii=False) + "\n")

def normalize_space(text):
  return re.sub(r"\s+", " ", str(text or "")).strip()

In [ ]:
# Cell 4: 数据读取与类别分布
train_rows = read_jsonl(TRAIN_DIR)
dev_rows = read_jsonl(DEV_DIR)

print("train:", len(train_rows))
print("dev:", len(dev_rows))

train_counter = Counter(row["Value"] for row in train_rows)
dev_counter = Counter(row["Value"] for row in dev_rows)

for label in LABELS:
  print(f"{label:28s} train={train_counter[label]:3d} dev={dev_counter[label]:3d}")

train: 3520
dev: 514
Self-direction–thought       train=119 dev= 17
Self-direction–action        train=124 dev= 18
Stimulation                  train=400 dev= 58
Hedonism                     train=164 dev= 24
Achievement                  train=174 dev= 26
Power–dominance              train=156 dev= 23
Power–resources              train=237 dev= 35
Face                         train=258 dev= 37
Security–personal            train=202 dev= 29
Security–societal            train= 70 dev= 11
Tradition                    train= 90 dev= 14
Conformity–rules             train=385 dev= 55
Conformity–interpersonal     train=236 dev= 34
Humility                     train=100 dev= 15
Benevolence–dependability    train=189 dev= 27
Benevolence–caring           train=317 dev= 46
Universalism–concern         train=160 dev= 24
Universalism–nature          train= 71 dev= 11
Universalism–tolerance       train= 68 dev= 10


In [ ]:
# Cell 5: Track 2 prompt 构造
TASK_INSTRUCTION = (
    "You are given a scenario, a question, and a target human value. "
    "Generate one concise, meaningful response that answers the question, "
    "fits the scenario, and naturally aligns with the target value."
)

def build_prompt(row, use_value_definition=True):
  scenario = normalize_space(row.get("Scenario", ""))
  question = normalize_space(row.get("Question", ""))
  value = normalize_space(row.get("Value", ""))
  parts = [
      TASK_INSTRUCTION,
      f"Scenario:\n{scenario}",
      f"Question:\n{question}",
      f"Target value:\n{value}",
  ]
  if use_value_definition:
    parts.append(f"Target value definition:\n{VALUE_DEFINITIONS.get(value,'')}")
  parts.append("Response:")
  return "\n\n".join(parts) + "\n"

sample = train_rows[0]
print(build_prompt(sample))
print("Gold response:", sample["Consistent Value Response"])
print("Contrastive  :", sample["Contrastive Response"])


You are given a scenario, a question, and a target human value. Generate one concise, meaningful response that answers the question, fits the scenario, and naturally aligns with the target value.

Scenario:
You work in a corporate setting where your manager frequently imposes strict guidelines.

Question:
How would you handle a disagreement with a superior during a team meeting?

Target value:
Conformity–interpersonal

Target value definition:
Avoidance of upsetting or harming other people.

Response:

Gold response: I would suggest alternative solutions diplomatically, ensuring my input is heard without challenging the manager openly, prioritizing team harmony and respect for hierarchy.
Contrastive  : I would publicly challenge the manager's decision, emphasizing my personal principles over maintaining workplace relationships, even if it risks disapproval or tension.


In [ ]:
# Cell 6: 构造 SFT 与 DPO 训练文件
# sft数据格式:prompt + consistent response(label) + text
def make_sft_rows(rows):
  data = []
  for row in rows:
    prompt = build_prompt(row)
    response = normalize_space(row["Consistent Value Response"])
    data.append({"prompt":prompt, "response":response, "text":prompt + response})
  return data

# dpo数据格式:prompt + chosen + rejected
def make_dpo_rows(rows):
  data = []
  for row in rows:
    prompt = build_prompt(row)
    chosen = normalize_space(row["Consistent Value Response"])
    rejected = normalize_space(row["Contrastive Response"])
    data.append({
        "prompt" : prompt,
        "chosen" : chosen,
        "rejected" : rejected,
    })
  return data
# Direct Preference Optimization
sft_train = make_sft_rows(train_rows)
sft_dev = make_sft_rows(dev_rows)

dpo_train = make_dpo_rows(train_rows)
dpo_dev = make_dpo_rows(dev_rows)

write_jsonl(DATA_DIR / "track2" / "sft_train.jsonl", sft_train)
write_jsonl(DATA_DIR / "track2" / "sft_dev.jsonl", sft_dev)
write_jsonl(DATA_DIR / "track2" / "dpo_train.jsonl", dpo_train)
write_jsonl(DATA_DIR / "track2" / "dpo_dev.jsonl", dpo_dev)


In [ ]:
# Cell 7: 随机查看几个训练样本

for row in random.sample(train_rows, 3):
    print("=" * 100)
    print("Value:", row["Value"])
    print("Scenario:", row["Scenario"])
    print("Question:", row["Question"])
    print("Consistent:", row["Consistent Value Response"])
    print("Contrastive:", row["Contrastive Response"])

Value: Achievement
Scenario: You are a professional athlete preparing for a major competition. You are considering whether to stick with your proven training regimen or experiment with a new, untested strategy that could either significantly improve your performance or increase your risk of injury.
Question: Which approach would you prioritize to maximize your chances of standing out in the competition?
Consistent: I would opt for the new strategy, as it offers the potential for greater competitive success and helps me stand out from other competitors who rely on conventional training methods.
Contrastive: I would stick to my proven training plan to minimize injury risks and ensure consistent results, prioritizing reliability and safety over the uncertain potential of higher achievement.
Value: Universalism–concern
Scenario: As a community leader, you are organizing an art installation for social justice. Local authorities are skeptical, arguing that the project doesn't help the econom